# الفصل 39: الصوت والكلام متعدد الوسائط

> الصوت هو الوسيط الثالث الأكثر أهمية بعد النص والصورة، وأصبح جزءًا أصيلًا من أنظمة الذكاء الاصطناعي التفاعلية (GPT-4o الصوتي، Gemini Live).

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 39
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 38: نماذج توليد الصور: Diffusion
- **الوسوم:** Whisper، TTS، EnCodec، Audio Tokenization

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch39-speech-audio.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch39-speech-audio.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> الصوت هو الوسيط الثالث الأكثر أهمية بعد النص والصورة، وأصبح جزءًا أصيلًا من أنظمة الذكاء الاصطناعي التفاعلية (GPT-4o الصوتي، Gemini Live).

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 38: نماذج توليد الصور: Diffusion


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم تمثيل الصوت — من الموجة إلى الرموز
- فهم معمارية Whisper (OpenAI) — التعرّف على الكلام كـ Seq2Seq عادي
- فهم Whisper


## 1. لماذا يهمّك هذا الموضوع؟

الصوت هو الوسيط الثالث الأكثر أهمية بعد النص والصورة، وأصبح جزءًا أصيلًا من أنظمة الذكاء الاصطناعي التفاعلية (GPT-4o الصوتي، Gemini Live). فهم كيفية تمثيل الصوت كرموز (tokens) هو المفتاح لدمجه مع LLMs بنفس الطريقة التي دمجنا بها الصور.


## 2. الخريطة الذهنية


```text
الصوت = موجة ضغط هواء مستمرة
│
├── التمثيل
│   ├── Waveform: 16kHz، عيّنة كل 1/16000 ثانية
│   ├── STFT/Mel: إطارات 25ms، خطوة 10ms، 80 mel bin
│   └── Tokens: EnCodec/SoundStream عبر RVQ
│
├── المهام
│   ├── ASR: صوت → نص (Whisper Encoder-Decoder)
│   ├── TTS: نص → صوت (Tacotron/FastSpeech/VITS)
│   └── Speech-to-Speech: صوت → صوت (Moshi/GPT-4o)
│
├── الدمج مع LLM
│   ├── Cascade: ASR → LLM → TTS (زمن مرتفع)
│   ├── Encoder + Adapter: Whisper → LLM (فهم)
│   └── Discrete tokens: صوت كرموز في نفس vocab (توحيد)
│
└── RVQ
    ├── codebook 1: المعلومة الخشنة
    ├── codebook 2: تصحيح الخطأ
    └── … حتى 8-32 codebook
```


## 3. الشرح المتدرج

### 🟢 أ) تمثيل الصوت — من الموجة إلى الرموز


```text
موجة صوتية خام (raw waveform, 16kHz-44.1kHz)
        │
        ▼ [تحويل فورييه قصير المدى STFT]
Mel-Spectrogram (تمثيل ثنائي الأبعاد: زمن × تردد)
        │
        ├──[مسار Whisper]──> Encoder-Decoder Transformer عادي (مثل T5)
        │
        └──[مسار الترميز العصبي]──> رموز صوتية منفصلة (Discrete Audio Tokens)
                                      عبر EnCodec / SoundStream (RVQ)
```


### 🟡 ب) معمارية Whisper (OpenAI) — التعرّف على الكلام كـ Seq2Seq عادي

Whisper يُعامِل التعرّف على الكلام كمسألة **ترجمة تسلسل إلى تسلسل** عادية تمامًا — بالضبط نفس معمارية Encoder-Decoder من الفصل 11!


In [ ]:
import torch
import torch.nn as nn

class WhisperStyleEncoder(nn.Module):
    """
    مُرمِّز صوتي: يأخذ Mel-Spectrogram ويُخرج تمثيلات سياقية
    يستخدم Conv1D أولاً لتقليل الطول الزمني، ثم Transformer Encoder عادي
    """
    def __init__(self, n_mels=80, d_model=512, n_layers=6, n_heads=8):
        super().__init__()
        self.conv1 = nn.Conv1d(n_mels, d_model, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1)
        self.pos_embed = nn.Parameter(torch.randn(1500, d_model))  # أقصى طول زمني

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, mel_spectrogram):
        # mel_spectrogram: [batch, n_mels, time]
        x = torch.nn.functional.gelu(self.conv1(mel_spectrogram))
        x = torch.nn.functional.gelu(self.conv2(x))          # تقليل الطول الزمني 2x
        x = x.permute(0, 2, 1)                                # [batch, time/2, d_model]
        x = x + self.pos_embed[:x.shape[1]]
        return self.encoder(x)

class WhisperStyleDecoder(nn.Module):
    """مُفكِّك نصّي عادي (مثل GPT) لكن مع Cross-Attention على مخرجات المُرمِّز الصوتي"""
    def __init__(self, vocab_size, d_model=512, n_layers=6, n_heads=8):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, target_ids, encoder_output, causal_mask):
        x = self.token_embed(target_ids)
        out = self.decoder(tgt=x, memory=encoder_output, tgt_mask=causal_mask)
        return self.output_proj(out)


> 💡 **تدريب متعدد المهام (Multitask)**: Whisper يُدرَّب على نفس المعمارية لأداء مهام متعددة بتوجيه عبر رموز خاصة في بداية التسلسل: `<transcribe>`, `<translate>`, `<lang:ar>`, وهذا نمط مهم جدًا في كل أنظمة Multimodal الحديثة — **تحويل كل المهام لصيغة نصية موحّدة** (نفس فلسفة T5 من الفصل 12!).

### 🔴 ج) طبقة التعمّق: ترميز الصوت المنفصل (Discrete Audio Tokenization) — EnCodec/SoundStream

لدمج الصوت مباشرة داخل LLM بنفس طريقة النص، نحتاج تحويله إلى **رموز منفصلة** (مثل رموز BPE). هذا ما تفعله **Residual Vector Quantization (RVQ)**:


In [ ]:
class ResidualVectorQuantizer(nn.Module):
    """
    RVQ: يستخدم عدة codebooks متتالية، كل واحد يُرمِّز "البقايا" (residual)
    من الـ codebook السابق — يسمح بدقة عالية بعدد رموز معقول
    """
    def __init__(self, num_quantizers=8, codebook_size=1024, dim=256):
        super().__init__()
        self.quantizers = nn.ModuleList([
            nn.Embedding(codebook_size, dim) for _ in range(num_quantizers)
        ])

    def forward(self, z):  # z: [batch, time, dim] — مخرج من مُرمِّز صوتي عصبي (CNN)
        residual = z
        quantized_out = torch.zeros_like(z)
        all_indices = []

        for codebook in self.quantizers:
            # إيجاد أقرب متجه في الـ codebook لكل موضع زمني
            distances = torch.cdist(residual, codebook.weight.unsqueeze(0).expand(z.shape[0], -1, -1))
            indices = distances.argmin(dim=-1)              # [batch, time]
            quantized = codebook(indices)                    # [batch, time, dim]

            quantized_out = quantized_out + quantized
            residual = residual - quantized                  # الباقي يُرمَّز بالـ codebook التالي
            all_indices.append(indices)

        return quantized_out, torch.stack(all_indices, dim=-1)  # [batch, time, num_quantizers]


🟢 **الحدس**: بدلاً من ضغط كل المعلومة في codebook واحد (يتطلب حجمًا هائلاً لدقة عالية)، RVQ يُقسّم المشكلة: الـ codebook الأول يلتقط المعلومة "الخشنة"، والثاني يصحح الخطأ المتبقي، وهكذا — تمامًا مثل الترميز بالأرقام العشرية (الخانة الأولى تقريب خشن، كل خانة لاحقة تحسين).

بمجرد حصولك على رموز صوتية منفصلة (مثل `[342, 891, 12, ...]` لكل إطار زمني × 8 codebooks)، يمكنك ببساطة **توسيع vocabulary الخاص بـ LLM** لتشمل هذه الرموز الصوتية، وتدريبه ليتنبأ بها بنفس طريقة النص تمامًا — هذا بالضبط نهج نماذج مثل **AudioLM** و**SpeechGPT** و**Moshi** (Kyutai).


## 4. مثال محلول خطوة بخطوة

لنحسب عدد الرموز الصوتية في ثانية واحدة. إشارة بمعدل 16kHz، إطار 25ms وخطوة 10ms:

- عدد العينات في الإطار: $16000 \times 0.025 = 400$ عيّنة.
- عدد الإطارات في الثانية: $1 / 0.010 = 100$ إطار.
- إذا استخدم EnCodec معدل 75 رمزًا/ثانية و8 codebooks، فعدد الرموز في الثانية = $75 \times 8 = 600$ رمز.

**المقارنة:** ثانية صوت واحدة تستهلك 600 رمز، بينما جملة عربية قصيرة (10 كلمات) قد تستهلك 30-60 رمزًا نصيًا. هذا يفسّر لماذا يكون الصوت مكلفًا في السياق، ولماذا نحتاج ضغطًا/تجزئة أذكى.

**مثال Whisper:** يُحوَّل الصوت إلى log-Mel spectrogram (80 قناة)، ثم يمر عبر Encoder، ويُولَّد النص عبر Decoder مع رموز مهمة مثل `<|transcribe|>` و`<|ar|>`.


## 5. نصائح وحيل

> 💡 **وحّد معدل العيّنة دائمًا**: 16kHz هو المعيار لمعظم نماذج ASR؛ خلط 44.1kHz و16kHz دون إعادة تشكيل يُنتج نتائج سيئة.

> 💡 **استخدم VAD قبل ASR**: اكتشاف النشاط الصوتي يقلّل التكلفة ويمنع تقسيم الكلام على الصمت.

> 💡 **مرّر لغة الجلسة صراحةً**: Whisper يدعم `<|ar|>`؛ تحديد اللغة يقلّل الهلوسة في المقاطع القصيرة.

> 💡 **اختبر على اللهجات لا على الفصحى فقط**: أداء Whisper على العربية الفصحى أعلى عادةً من اللهجات؛ ابنِ مجموعة اختبار لهجية صغيرة.

> 💡 **قِس WER وCER معًا**: WER يعاقب أخطاء الكلمات، CER يفيد أكثر في العربية بسبب الإملاء والتشكيل.


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاعتماد على Cascade في محادثة فورية**: ASR → LLM → TTS يضيف زمنًا تراكميًا ويفقد النبرة والتوقفات. للمحادثة الحقيقية فضّل Speech-to-Speech أو streaming.

> ⚠️ **تجاهل إعادة تشكيل الصوت (Resampling)**: نموذج دُرِّب على 16kHz لن يعمل جيدًا على 8kHz أو 48kHz دون معالجة.

> ⚠️ **codebook واحد كبير بدل RVQ**: البحث عن أقرب جار يصبح مكلفًا، وتوزيع الرموز يصبح نادرًا جدًا، فيصعب التدريب.

> ⚠️ **الحكم على ASR من ملف نظيف واحد**: الضوضاء، اللهجة، والميكروفون يغيّرون النتائج drastically. اختبر ظروفًا واقعية.

> ⚠️ **نسيان الخصوصية**: الصوت بيانات بيومترية حساسة؛ لا تخزّنه أو ترسله دون تشفير وموافقة.


## 7. جدول مقارنة

| الأسلوب | أمثلة | المبدأ | الميزة |
|---|---|---|---|
| **Cascade (تتابعي)** | ASR→LLM→TTS منفصلة | ثلاث أنظمة مستقلة | بساطة، سهولة الصيانة |
| **Encoder-Decoder مباشر** | Whisper | صوت↔نص عبر Transformer عادي | دقة عالية للتفريغ |
| **Discrete Tokens Unified** | AudioLM, Moshi, SpeechGPT | رموز صوتية داخل vocabulary LLM | زمن استجابة منخفض، تعبير صوتي (نبرة) محفوظ |
| **Continuous Embedding Fusion** | Qwen-Audio | تضمينات مستمرة + Cross-Attention | يحافظ على تفاصيل دقيقة |

> ⚠️ **خطأ شائع**: استخدام نهج Cascade (ASR→LLM→TTS) في تطبيقات المحادثة الصوتية الفورية يُضيف زمن استجابة تراكمي كبير (latency من 3 أنظمة منفصلة) ويفقد المعلومات شبه اللفظية (نبرة الصوت، التنهد، التوقفات) التي لا يحملها النص المُفرَّغ — لهذا تتجه الأنظمة الحديثة (Moshi, GPT-4o) لنمذجة الصوت **مباشرة** دون المرور عبر نص وسيط.

---


## 8. ملخص أفضل الممارسات

- ✅ ابدأ بـ Whisper أو نموذج ASR مُدرَّب مسبقًا، ولا تدرّب ASR من الصفر إلا لمجال/لغة خاصة.
- ✅ استخدم VAD وstreaming لتقليل الزمن والتكلفة في التطبيقات التفاعلية.
- ✅ وحّد معدل العيّنة واختبر اللهجات والضوضاء قبل النشر.
- ✅ قِس WER/CER على مجموعة اختبار عربية حقيقية، لا على عيّنة إنجليزية مترجمة.
- ✅ صمّم سياسة خصوصية واضحة للصوت (تشفير، حذف، موافقة).


## 9. أسئلة مقابلات

**س1: لماذا لا يكفي codebook واحد كبير لترميز الصوت؟**

**الإجابة النموذجية:** للحصول على دقة إعادة بناء عالية بـ codebook واحد، نحتاج ملايين الإدخالات، فيصبح البحث عن أقرب جار بطيئًا، ويصبح توزيع الرموز نادرًا جدًا (معظم الرموز لا تُستخدم)، مما يُضعف التدرجات. RVQ يحل ذلك بتقسيم المعلومة: codebook أول خشن، ثم codebooks تصحح الباقي، فيبقى حجم كل codebook معقولًا والتدريب أسهل.

**س2: ما الفرق بين Whisper وEnCodec في تمثيل الصوت؟**

**الإجابة النموذجية:** Whisper نموذج ASR Encoder-Decoder يُخرج نصًا، وتمثيله الداخلي مستمر (log-Mel + Transformer). EnCodec مُرمِّز صوتي عام يُخرج رموزًا منفصلة عبر RVQ، ويمكن استخدامه لدمج الصوت في LLM أو لإعادة توليد الصوت. الأول للفهم، الثاني للتمثيل/التوليد.

**س3: متى تختار Speech-to-Speech بدلًا من Cascade؟**

**الإجابة النموذجية:** عندما يكون الزمن الحقيقي والنبرة العاطفية مهمين (مساعد صوتي، محادثة)، لأن Cascade يفقد المعلومات شبه اللفظية ويجمع زمن ثلاثة أنظمة. Cascade قد يكون أفضل عندما تحتاج شفافية/تسجيلًا نصيًا أو عندما لا يتوفر نموذج S2S قوي للغة.


## 10. تمرين عملي

1. حمّل `openai/whisper-base` وفرّغ ملفًا عربيًا قصيرًا، ثم احسب WER يدويًا مقابل تفريغ مرجعي.
2. كرّر التجربة بعد إضافة ضوضاء خلفية بمستوى 10dB، وقارن التدهور.
3. احسب عدد الرموز الصوتية لثانية واحدة باستخدام EnCodec (75 token/s × 8 codebooks) وقارنه بعدد رموز النص لنفس المقطع.
4. صمّم مخططًا بسيطًا لنظام محادثة صوتية: VAD → ASR → LLM → TTS، وحدّد أين يمكن تقليل الزمن.
5. اكتب فقرة عن مخاطر الخصوصية في تخزين الصوت وحلولها.


## 11. مراجع للتعمّق

- Radford, A., et al. (2022). *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper).
- Défossez, A., et al. (2022). *High Fidelity Neural Audio Compression* (EnCodec).
- Borsos, Z., et al. (2022). *AudioLM: A Language Modeling Approach to Audio Generation*.
- Défossez, A., et al. (2024). *Moshi: A Speech-Text Foundation Model for Real-Time Dialogue*.
- Gong, Y., et al. (2023). *Joint Speech and Text Machine Translation*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 39: الصوت والكلام متعدد الوسائط.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
